In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
dataset=pd.read_csv('Social_Network_Ads.csv')

In [3]:
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [4]:
dataset=pd.get_dummies(dataset,dtype=int,drop_first=True)

In [5]:
dataset

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,1
1,15810944,35,20000,0,1
2,15668575,26,43000,0,0
3,15603246,27,57000,0,0
4,15804002,19,76000,0,1
...,...,...,...,...,...
395,15691863,46,41000,1,0
396,15706071,51,23000,1,1
397,15654296,50,20000,1,0
398,15755018,36,33000,0,1


In [6]:
dataset.columns

Index(['User ID', 'Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [7]:
independent=dataset[['Age', 'EstimatedSalary','Gender_Male']]
dependent=dataset['Purchased']

In [8]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(independent,dependent,test_size=1/3,random_state=0)

In [9]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [10]:
from sklearn.tree import DecisionTreeClassifier

In [11]:
from sklearn.model_selection import GridSearchCV

param_grid = {'criterion':['gini','entropy'],
              'max_features':['auto','sqrd','log2'],
              'splitter':['best','random']}

grid = GridSearchCV(DecisionTreeClassifier(),param_grid,refit=True,verbose=3,n_jobs=-1,scoring='f1_weighted')

grid.fit(x_train,y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


C:\Users\lenovo\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
40 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
9 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\lenovo\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lenovo\anaconda3\Lib\site-packages\sklearn\base.py", line 1382, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\lenovo\anaconda3\Lib\site-packages\sklearn\base.py", line 436, in _validate_params
 

GridSearchCV(estimator=DecisionTreeClassifier(), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_features': ['auto', 'sqrd', 'log2'],
                         'splitter': ['best', 'random']},
             scoring='f1_weighted', verbose=3)

In [12]:
re=grid.cv_results_
grid_predictions = grid.predict(x_test)

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test,grid_predictions)

from sklearn.metrics import classification_report
clf_report = classification_report(y_test,grid_predictions)

In [13]:
from sklearn.metrics import f1_score
f1_macro = f1_score(y_test,grid_predictions,average='weighted')
print("The f1_macro value for best parameter{}:".format(grid.best_params_),f1_macro)

The f1_macro value for best parameter{'criterion': 'gini', 'max_features': 'log2', 'splitter': 'best'}: 0.8866497008047322


In [14]:
print("The confusion Matrix:\n",cm)

The confusion Matrix:
 [[80  5]
 [10 39]]


In [15]:
print("The report:\n",clf_report)

The report:
               precision    recall  f1-score   support

           0       0.89      0.94      0.91        85
           1       0.89      0.80      0.84        49

    accuracy                           0.89       134
   macro avg       0.89      0.87      0.88       134
weighted avg       0.89      0.89      0.89       134



In [16]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test,grid.predict_proba(x_test)[:,1])

np.float64(0.868547418967587)

In [17]:
table=pd.DataFrame.from_dict(re)

In [18]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.003001,0.000652,0.000000,0.000000,gini,auto,best,"{'criterion': 'gini', 'max_features': 'auto', ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
1,0.003677,0.001512,0.000000,0.000000,gini,auto,random,"{'criterion': 'gini', 'max_features': 'auto', ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
2,0.002304,0.000748,0.000000,0.000000,gini,sqrd,best,"{'criterion': 'gini', 'max_features': 'sqrd', ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
3,0.004008,0.003341,0.000000,0.000000,gini,sqrd,random,"{'criterion': 'gini', 'max_features': 'sqrd', ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
4,0.016990,0.008548,0.019220,0.004398,gini,log2,best,"{'criterion': 'gini', 'max_features': 'log2', ...",0.826263,0.868752,0.814409,0.886792,0.943041,0.867852,0.046041,1
5,0.008990,0.004212,0.013257,0.002746,gini,log2,random,"{'criterion': 'gini', 'max_features': 'log2', ...",0.847141,0.868752,0.759244,0.850543,0.867097,0.838556,0.040581,3
6,0.002157,0.000419,0.000000,0.000000,entropy,auto,best,"{'criterion': 'entropy', 'max_features': 'auto...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
7,0.001897,0.000407,0.000000,0.000000,entropy,auto,random,"{'criterion': 'entropy', 'max_features': 'auto...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
8,0.001912,0.000375,0.000000,0.000000,entropy,sqrd,best,"{'criterion': 'entropy', 'max_features': 'sqrd...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
9,0.001762,0.000528,0.000000,0.000000,entropy,sqrd,random,"{'criterion': 'entropy', 'max_features': 'sqrd...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,5


In [19]:
age_input=float(input("Age:"))
estimatedsalary_input=float(input("EstimatedSalary:"))
gender_male_input=float(input("Gender_Male:"))              

Age: 19
EstimatedSalary: 19000
Gender_Male: 1


In [20]:
Future_Prediction=grid.predict([[age_input,estimatedsalary_input,gender_male_input]])
print("Future_Prediction={}".format(Future_Prediction))

Future_Prediction=[1]
